In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [3]:
df = pd.read_csv("retail best dataset.csv", encoding='latin-1')

In [4]:
df.head(3)

,Order_ID,Order_Date,Store_ID,Store_City,Store_Region,Product_ID,Product_Name,Category,Sub_Category,Customer_ID,Customer_Name,Segment,Salesperson,Quantity,Unit_Price,Discount,Sales,Cost,Profit,Payment_Method
0,ORD001,1/5/2024,ST001,New York,West,PRD101,Samsung Galaxy S23,Electronics,Smartphones,CUST1001,Rajesh Kumar,Consumer,John Smith,2,75000,5,142500,120000,22500,Card
1,ORD002,1/5/2024,ST001,New York,West,PRD102,Sony Headphones,Electronics,Accessories,CUST1002,Priya Patel,Consumer,John Smith,3,3500,0,10500,7000,3500,Cash
2,ORD003,1/5/2024,ST002,Los Angeles,North,PRD201,Levis Jeans,Clothing,Men's Wear,CUST1003,Amit Singh,Consumer,Emily Johnson,4,2500,10,9000,6000,3000,Card


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Order_ID        1455 non-null   object
 1   Order_Date      1455 non-null   object
 2   Store_ID        1455 non-null   object
 3   Store_City      1455 non-null   object
 4   Store_Region    1455 non-null   object
 5   Product_ID      1455 non-null   object
 6   Product_Name    1455 non-null   object
 7   Category        1455 non-null   object
 8   Sub_Category    1455 non-null   object
 9   Customer_ID     1455 non-null   object
 10  Customer_Name   1455 non-null   object
 11  Segment         1455 non-null   object
 12  Salesperson     1455 non-null   object
 13  Quantity        1455 non-null   int64 
 14  Unit_Price      1455 non-null   int64 
 15  Discount        1455 non-null   int64 
 16  Sales           1455 non-null   int64 
 17  Cost            1455 non-null   int64 
 18  Profit  

In [18]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce', format='%m/%d/%Y')

### Primary KPIs

In [50]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order_ID'].nunique()
profit_margin = (total_profit / total_sales) * 100

kpi_data = {
    'Total Sales': f"${total_sales/1000000:.2f}M",
    'Total Profit': f"${total_profit/1000000:.2f}M", 
    'Orders': f"{total_orders:,}",
    'Profit Margin %': f"{profit_margin:.2f}%"
}

kpi_df = pd.DataFrame([kpi_data])
print(kpi_df.to_string(index=False))

Total Sales Total Profit Orders Profit Margin %
    $33.85M       $7.76M  1,378          22.93%


### Monthly Revenue Trends

In [48]:
# Revenue by Month
gb_rev = df.groupby(df['Order_Date'].dt.month_name().str[:3])['Sales'].sum().reset_index()
gb_rev.columns = ['Month', 'Revenue']

# Cost by Month  
gb_cost = df.groupby(df['Order_Date'].dt.month_name().str[:3])['Cost'].sum().reset_index()
gb_cost.columns = ['Month', 'Cost']

sort_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Merge both dataframes
gb = pd.merge(gb_rev, gb_cost, on='Month')
gb['Month'] = pd.Categorical(gb['Month'], categories=sort_months, ordered=True)
gb = gb.sort_values('Month')

fig = go.Figure()

# Revenue line
fig.add_trace(go.Scatter(
    x=gb['Month'], y=gb['Revenue'],
    mode='lines+markers', line_shape='spline',
    name='Revenue', line=dict(width=3, color='royalblue'),
    marker=dict(size=10, color='royalblue')
))

# Cost line  
fig.add_trace(go.Scatter(
    x=gb['Month'], y=gb['Cost'],
    mode='lines+markers', line_shape='spline',
    name='Cost', line=dict(width=3, color='#FF6B4A'),
    marker=dict(size=10, color='#FF6B4A')
))

# Annotations for Revenue
for i, row in gb.iterrows():
    fig.add_annotation(
        x=row['Month'], y=row['Revenue'],
        text=f"${row['Revenue']/1000000:.1f}M",
        showarrow=False, yshift=15,
        font=dict(size=10, color='royalblue')
    )
    
    fig.add_annotation(
        x=row['Month'], y=row['Cost'],
        text=f"${row['Cost']/1000000:.1f}M", 
        showarrow=False, yshift=-20,
        font=dict(size=10, color='#FF6B4A')
    )

fig.update_layout(
    title='<b>Monthly Revenue vs Cost Trends</b>',
    xaxis_title='', yaxis_title='Amount (in Millions)',
    plot_bgcolor='white', hovermode='x unified',
    title_x=0.5, title_font_size=18,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.add_annotation(
    x=0.98, y=0.98, xref='paper', yref='paper',
    text=f'<b>Total Revenue: ${gb["Revenue"].sum()/1000000:.1f}M<br>Total Cost: ${gb["Cost"].sum()/1000000:.1f}M</b>',
    showarrow=False, font=dict(size=12, color='white'),
    bgcolor='#2E4057', bordercolor='#1E2B3A', borderwidth=2,
    align='center', borderpad=6
)

fig.update_yaxes(tickprefix='$', ticksuffix='M', tickformat=',.1f')
fig.show()

### Monthly Profit Trends

In [46]:
# Profit by Month
gb = df.groupby(df['Order_Date'].dt.month_name().str[:3])['Profit'].sum().reset_index()
gb.columns = ['Month', 'Total_Profit']

sort_months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
gb['Month'] = pd.Categorical(gb['Month'], categories=sort_months, ordered=True)
gb = gb.sort_values('Month')

fig = px.line(gb, x='Month', y='Total_Profit', title='<b>Monthly Profit Trends</b>',
              markers=True, line_shape='spline')

for i, row in gb.iterrows():
    fig.add_annotation(
        x=row['Month'],
        y=row['Total_Profit'],
        text=f"${row['Total_Profit']/1000000:.2f}M",
        showarrow=False,
        yshift=15,
        font=dict(size=11)
    )

fig.update_layout(
    xaxis_title='',
    yaxis_title='Total Profit (in Millions)',
    plot_bgcolor='white',
    hovermode='x unified',
    title_x=0.5,
    title_font_size=18
)

fig.add_annotation(
    x=0.5,
    y=0.98,
    xref='paper',
    yref='paper',
    text=f'<b>Total Profit: ${gb["Total_Profit"].sum()/1000000:.1f}M</b>',
    showarrow=False,
    font=dict(size=14, color='white'),
    bgcolor='#FF6B4A',
    bordercolor='#CC4A2F',
    borderwidth=2
)

fig.update_traces(line=dict(width=3, color='#FF6B4A'),
                  marker=dict(size=10, color='#CC4A2F'))

fig.update_yaxes(tickprefix='$', ticksuffix='M', tickformat=',.1f')
fig.show()

### Category by Revenue

In [51]:

category_rev = df.groupby('Category')['Sales'].sum().reset_index()
category_rev.columns = ['Category', 'Total_Revenue']

fig = go.Figure(data=[go.Pie(
    labels=category_rev['Category'],
    values=category_rev['Total_Revenue'],
    hole=0.5,
    textinfo='label+percent',
    textposition='outside',
    marker=dict(colors=['#FF6B4A', '#4A90E2', '#50C878', '#FFD700', '#9B59B6'])
)])

fig.update_layout(
    title='<b>Category Revenue Distribution</b>',
    title_x=0.5,
    title_font_size=18,
    showlegend=False,
    annotations=[dict(
        text=f'<b>Total Revenue<br>${category_rev["Total_Revenue"].sum()/1000000:.1f}M</b>',
        x=0.5, y=0.5,
        font_size=14,
        showarrow=False
    )]
)

fig.show()

### Store City by Sales

In [52]:

city_sales = df.groupby('Store_City')['Sales'].sum().reset_index()
city_sales.columns = ['Store_City', 'Total_Revenue']
city_sales = city_sales.sort_values('Total_Revenue', ascending=True)

fig = px.bar(city_sales, 
             x='Total_Revenue', 
             y='Store_City',
             orientation='h',
             title='<b>Store City Revenue</b>',
             text=city_sales['Total_Revenue'].apply(lambda x: f'${x/1000000:.1f}M'))

fig.update_traces(
    marker_color='royalblue',
    textposition='outside',
    textfont_size=11
)

fig.update_layout(
    xaxis_title='Total Revenue (in Millions)',
    yaxis_title='',
    plot_bgcolor='white',
    title_x=0.5,
    title_font_size=18,
    height=400 + (len(city_sales) * 20),
    xaxis=dict(tickprefix='$', ticksuffix='M')
)

fig.show()

### Payment Distribution by Sales

In [53]:

payment_sales = df.groupby('Payment_Method')['Sales'].sum().reset_index()
payment_sales.columns = ['Payment_Method', 'Total_Revenue']

fig = go.Figure(data=[go.Pie(
    labels=payment_sales['Payment_Method'],
    values=payment_sales['Total_Revenue'],
    hole=0.5,
    textinfo='label+percent',
    textposition='outside',
    marker=dict(colors=['#FF6B4A', '#4A90E2', '#50C878', '#FFD700', '#9B59B6', '#FF7F50'])
)])

fig.update_layout(
    title='<b>Payment Method Distribution</b>',
    title_x=0.5,
    title_font_size=18,
    showlegend=False,
    annotations=[dict(
        text=f'<b>Total Revenue<br>${payment_sales["Total_Revenue"].sum()/1000000:.1f}M</b>',
        x=0.5, y=0.5,
        font_size=14,
        showarrow=False
    )]
)

fig.show()

### Region Distribution

In [54]:

region_data = df.groupby('Store_Region').agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).reset_index()
region_data.columns = ['Store_Region', 'Total_Revenue', 'Total_Profit']

fig = go.Figure(data=[
    go.Bar(
        name='Revenue',
        x=region_data['Store_Region'],
        y=region_data['Total_Revenue'],
        text=region_data['Total_Revenue'].apply(lambda x: f'${x/1000000:.1f}M'),
        textposition='outside',
        marker_color='royalblue'
    ),
    go.Bar(
        name='Profit',
        x=region_data['Store_Region'],
        y=region_data['Total_Profit'],
        text=region_data['Total_Profit'].apply(lambda x: f'${x/1000000:.1f}M'),
        textposition='outside',
        marker_color='#FF6B4A'
    )
])

fig.update_layout(
    title='<b>Region Wise Revenue & Profit</b>',
    xaxis_title='',
    yaxis_title='Amount (in Millions)',
    plot_bgcolor='white',
    title_x=0.5,
    title_font_size=18,
    barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.update_yaxes(tickprefix='$', ticksuffix='M', tickformat=',.1f')
fig.show()

### Category by Profit

In [55]:

category_profit = df.groupby('Category').agg({
    'Order_ID': 'count',
    'Profit': 'sum'
}).reset_index()
category_profit.columns = ['Category', 'Distribution', 'Total_Profit']
category_profit = category_profit.sort_values('Total_Profit', ascending=True)

fig = go.Figure()

# Adding bars
fig.add_trace(go.Bar(
    y=category_profit['Category'],
    x=category_profit['Total_Profit'],
    orientation='h',
    text=category_profit.apply(lambda row: f'${row["Total_Profit"]/1000000:.1f}M | Orders: {row["Distribution"]:,}', axis=1),
    textposition='outside',
    marker_color='#50C878',
    name='Profit'
))

fig.update_layout(
    title='<b>Category Profit Distribution</b>',
    xaxis_title='Total Profit (in Millions)',
    yaxis_title='',
    plot_bgcolor='white',
    title_x=0.5,
    title_font_size=18,
    height=400 + (len(category_profit) * 20),
    xaxis=dict(tickprefix='$', ticksuffix='M')
)

fig.update_traces(textfont_size=11)
fig.show()

### Yearly Revenue Trends

In [57]:
yearly_rev = df.groupby(df['Order_Date'].dt.year)['Sales'].sum().reset_index()
yearly_rev.columns = ['Year', 'Total_Revenue']
yearly_rev['Revenue_Growth'] = (yearly_rev['Total_Revenue'] / yearly_rev['Total_Revenue'].sum() * 100).round(2)

fig = go.Figure(data=[go.Pie(
    labels=yearly_rev['Year'].astype(str),
    values=yearly_rev['Total_Revenue'],
    hole=0.5,
    textinfo='label+percent',
    textposition='outside',
    text=yearly_rev.apply(lambda x: f'{x["Year"]}<br>${x["Total_Revenue"]/1000000:.1f}M<br>({x["Revenue_Growth"]:.1f}%)', axis=1),
    hoverinfo='label+percent+value',
    marker=dict(colors=['#FF6B4A', '#4A90E2', '#50C878', '#FFD700', '#9B59B6'])
)])

fig.update_layout(
    title='<b>Yearly Revenue Distribution</b>',
    title_x=0.5,
    title_font_size=18,
    showlegend=True,
    annotations=[dict(
        text=f'<b>Total Revenue<br>${yearly_rev["Total_Revenue"].sum()/1000000:.1f}M</b>',
        x=0.5, y=0.5,
        font_size=14,
        showarrow=False
    )]
)

fig.show()

### Sales by Customer Segment

In [58]:

segment_sales = df.groupby('Segment')['Sales'].sum().reset_index()
segment_sales.columns = ['Segment', 'Total_Revenue']

fig = go.Figure(data=[go.Pie(
    labels=segment_sales['Segment'],
    values=segment_sales['Total_Revenue'],
    hole=0.5,
    textinfo='label+percent',
    textposition='outside',
    marker=dict(colors=['#4A90E2', '#50C878', '#FF6B4A', '#9B59B6', '#FFD700'])
)])

fig.update_layout(
    title='<b>Customer Segment Revenue Distribution</b>',
    title_x=0.5,
    title_font_size=18,
    showlegend=False,
    annotations=[dict(
        text=f'<b>Total Revenue<br>${segment_sales["Total_Revenue"].sum()/1000000:.1f}M</b>',
        x=0.5, y=0.5,
        font_size=14,
        showarrow=False
    )]
)

fig.show()